# Calibrated Hallucination Detection in LLMs via Internal Uncertainty Signals

This notebook demonstrates the end-to-end hallucination detection pipeline:
1. **Data Loading & Generation**: Greedy answer + token logprobs + hidden states + stochastic sampling
2. **Labeling & Audit**: Normalized exact match / alias containment against gold answers
3. **Feature Extraction**: Gray-box (logprobs, entropy), Black-box (consistency, semantic entropy), White-box (layer hidden states)
4. **Probing & Layer Sweep**: Linear probe on internal activations across transformer layers
5. **Calibration & Evaluation**: Platt scaling, Isotonic regression, ECE, Brier score, and Risk-Coverage selective prediction


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from halludetect.config import load_config
from halludetect.data import split_by_question, generate_mock_dataset, load_dataframe
from halludetect.generate import get_generator
from halludetect.labeling import label_prediction, audit_labeling
from halludetect.features.grey import extract_graybox_features
from halludetect.features.consistency import extract_blackbox_features
from halludetect.features.probe import HiddenStateProbe, evaluate_layer_sweep
from halludetect.models import SupervisedDetector, ALL_FEATURES
from halludetect.calibration import CalibratedDetector
from halludetect.metrics import compute_all_metrics, compute_ece, compute_risk_coverage
from halludetect.plots import plot_reliability_diagram, plot_layer_sweep, plot_risk_coverage_curve

print('HalluDetect environment successfully loaded!')

## 1. Load Extracted Features & Inspect Uncertainty Signals

In [ ]:
df_feats = load_dataframe('../data/processed/features_triviaqa.parquet')
print(f'Total questions: {len(df_feats)}')
print('Label distribution (0=Correct, 1=Hallucinated):')
print(df_feats['label'].value_counts())
df_feats[['question', 'greedy_text', 'label', 'mean_logprob', 'mean_entropy', 'agreement_rate', 'semantic_entropy']].head()

## 2. Layer Sweep for White-Box Hidden-State Probing

In [ ]:
df_sweep = pd.read_csv('../results/tables/layer_sweep.csv')
records = df_sweep.to_dict(orient='records')
best_layer = records[np.argmax([r['auroc'] for r in records])]['layer']
fig_layer = plot_layer_sweep(records, best_layer=best_layer)
plt.show()

## 3. Reliability Diagrams: Before vs. After Calibration

In [ ]:
data = np.load('../data/processed/predictions_cache.npz', allow_pickle=True)
y_test = data['y_test']
rel_dict = {}
if 'pred__Baseline: Mean LogProb' in data:
    rel_dict['Raw Mean LogProb'] = data['pred__Baseline: Mean LogProb']
if 'pred__Combined: Gray+Black+Probe (LogReg)' in data:
    rel_dict['Combined (Raw)'] = data['pred__Combined: Gray+Black+Probe (LogReg)']
if 'cal__Combined: Gray+Black+Probe (LogReg) (Platt)' in data:
    rel_dict['Combined (Platt Calibrated)'] = data['cal__Combined: Gray+Black+Probe (LogReg) (Platt)']

fig_rel = plot_reliability_diagram(y_test, rel_dict)
plt.show()

## 4. Selective Prediction: Risk vs. Coverage Curves

In [ ]:
curve_dict = {
    k.replace('pred__', ''): data[k]
    for k in data.files if k.startswith('pred__')
}
fig_rc = plot_risk_coverage_curve(y_test, curve_dict)
plt.show()

## 5. Evaluation Metrics Summary Table

In [ ]:
df_metrics = pd.read_csv('../results/tables/evaluation_metrics.csv')
df_metrics